In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import input_file_name, current_timestamp, current_date,col

In [0]:
SOURCE_VOLUME_PATH = "/Volumes/dbr_dev/artemzharkov10_bronze/raw_data/gdelt_history/"
TARGET_TABLE = "dbr_dev.artemzharkov10_bronze"

gdelt_schema = StructType([
    StructField("GlobalEventID", IntegerType(), True),         # _c0
    StructField("SqlDate", IntegerType(), True),               # _c1
    StructField("MonthYear", IntegerType(), True),             # _c2
    StructField("Year", IntegerType(), True),                  # _c3
    StructField("FractionDate", DoubleType(), True),           # _c4
    
    StructField("Actor1Code", StringType(), True),             # _c5
    StructField("Actor1Name", StringType(), True),             # _c6
    StructField("Actor1CountryCode", StringType(), True),      # _c7
    StructField("Actor1KnownGroupCode", StringType(), True),   # _c8
    StructField("Actor1EthnicCode", StringType(), True),       # _c9
    StructField("Actor1Religion1Code", StringType(), True),    # _c10
    StructField("Actor1Religion2Code", StringType(), True),    # _c11
    StructField("Actor1Type1Code", StringType(), True),        # _c12
    StructField("Actor1Type2Code", StringType(), True),        # _c13
    StructField("Actor1Type3Code", StringType(), True),        # _c14
    
    StructField("Actor2Code", StringType(), True),             # _c15
    StructField("Actor2Name", StringType(), True),             # _c16
    StructField("Actor2CountryCode", StringType(), True),      # _c17
    StructField("Actor2KnownGroupCode", StringType(), True),   # _c18
    StructField("Actor2EthnicCode", StringType(), True),       # _c19
    StructField("Actor2Religion1Code", StringType(), True),    # _c20
    StructField("Actor2Religion2Code", StringType(), True),    # _c21
    StructField("Actor2Type1Code", StringType(), True),        # _c22
    StructField("Actor2Type2Code", StringType(), True),        # _c23
    StructField("Actor2Type3Code", StringType(), True),        # _c24
    
    StructField("IsRootEvent", IntegerType(), True),           # _c25
    StructField("EventCode", StringType(), True),              # _c26 
    StructField("EventBaseCode", StringType(), True),          # _c27
    StructField("EventRootCode", StringType(), True),          # _c28
    StructField("QuadClass", IntegerType(), True),             # _c29
    StructField("GoldsteinScale", DoubleType(), True),         # _c30
    StructField("NumMentions", IntegerType(), True),           # _c31
    StructField("NumSources", IntegerType(), True),            # _c32
    StructField("NumArticles", IntegerType(), True),           # _c33
    StructField("AvgTone", DoubleType(), True),                # _c34 
    
    StructField("Actor1Geo_Type", IntegerType(), True),        # _c35
    StructField("Actor1Geo_FullName", StringType(), True),     # _c36
    StructField("Actor1Geo_CountryCode", StringType(), True),  # _c37
    StructField("Actor1Geo_ADM1Code", StringType(), True),     # _c38
    StructField("Actor1Geo_ADM2Code", StringType(), True),     # _c39
    StructField("Actor1Geo_Lat", DoubleType(), True),          # _c40
    StructField("Actor1Geo_Long", DoubleType(), True),         # _c41
    StructField("Actor1Geo_FeatureID", StringType(), True),    # _c42
    
    StructField("Actor2Geo_Type", IntegerType(), True),        # _c43
    StructField("Actor2Geo_FullName", StringType(), True),     # _c44
    StructField("Actor2Geo_CountryCode", StringType(), True),  # _c45
    StructField("Actor2Geo_ADM1Code", StringType(), True),     # _c46
    StructField("Actor2Geo_ADM2Code", StringType(), True),     # _c47
    StructField("Actor2Geo_Lat", DoubleType(), True),          # _c48
    StructField("Actor2Geo_Long", DoubleType(), True),         # _c49
    StructField("Actor2Geo_FeatureID", StringType(), True),    # _c50
    
    StructField("ActionGeo_Type", IntegerType(), True),        # _c51
    StructField("ActionGeo_FullName", StringType(), True),     # _c52
    StructField("ActionGeo_CountryCode", StringType(), True),  # _c53
    StructField("ActionGeo_ADM1Code", StringType(), True),     # _c54
    StructField("ActionGeo_ADM2Code", StringType(), True),     # _c55
    StructField("ActionGeo_Lat", DoubleType(), True),          # _c56
    StructField("ActionGeo_Long", DoubleType(), True),         # _c57
    StructField("ActionGeo_FeatureID", StringType(), True),    # _c58
    
    StructField("DateAdded", IntegerType(), True),             # _c59
    StructField("SourceUrl", StringType(), True)               # _c60
])

df_bronze = (spark.read.format("csv")
  .option("delimiter","\t")
  .option("header", "true")
  .schema(gdelt_schema)
  .load(SOURCE_VOLUME_PATH)) #Spark see Unity Catalog and automation add hard path to azure storage

df_bronze_with_metadata = (
  df_bronze
    .withColumn("source_filename", col("_metadata.file_path"))
    .withColumn("ingest_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

(df_bronze_with_metadata.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("dbr_dev.artemzharkov10_bronze.gdlet_history_bronze"))

In [0]:
df_bronze = spark.read.table("dbr_dev.artemzharkov10_bronze.gdlet_history_bronze")

display(df_bronze.limit(30))